# HVAC 2.0 — Stage 3: No-Pretraining Control (4 configs / 40 runs)

Corresponds to `proposal/proposal.md` v6, §7, Stage 3. **The model is randomly initialized with no MIMII pretraining**
and trained directly on the self-recorded fan clips, to measure the value of pretraining itself for few-shot adaptation.

| Grid | Configs | LORO | Runs |
| --- | --- | --- | --- |
| Data level 5/10/20/40 × full training (only scheme) | 4 | 10 folds | 40 |

A randomly initialized model has no "learned encoder/decoder", so there is no layer-scheme variable — only "full".
Hyperparameters are identical to Stage 2: `lr=1e-4 / epochs=30 / batch=64`, no early stopping. Architecture identical to Stage 0.

**This stage does not need MIMII data**: the random-init model is trained and evaluated only on the self-recorded fan; there is no source domain,
so `Baseline/` is not read and `0_dB_fan.zip` is not extracted. Only the 180 target-domain clips are needed.

---

### Expected result (recorded as-is, not a bug)

With 5 clips (~1545 320-dim feature vectors) and no prior, 30 epochs of training will very likely
overfit until reconstruction error approaches zero, with discrimination near chance — direct evidence that pretraining has value.
The trivial floor of best F1 is **0.800** (obtained by flagging everything as anomalous at a 1:2 class ratio),
so F1 at this level will likely show ~0.80; **you must look at AUC** to judge real discriminative power.

### Resume from checkpoint

Same mechanism as Stage 2: source of truth is `Stage3/logs/stage3_runs.jsonl`, appended run by run; re-running resumes automatically.
40 runs is far smaller than Stage 2 and normally finishes in a few minutes, but the protection logic is kept — disconnects can happen anytime.

---

**How to run**: `Runtime → Run all`. **Runtime**: T4 GPU or CPU both work; the 40 runs themselves are fast
(measured on local CPU: 0.3~2.4 s per run); the bottleneck is Drive mounting and target-domain feature extraction.

**Embedded config** sha256 = `8430e581ae6b4e14...`, the same file as the previous three stages. Note: this is the hash of the original config file; translating its comments changes the hash, so a mismatch with this value is expected and harmless.

## 1 · Environment and dependencies

In [ ]:
import sys, os, re, time, json, glob, hashlib, zipfile, subprocess, traceback
from collections import defaultdict
from datetime import datetime, timedelta

def _ensure(pkg, imp=None):
    try:
        __import__(imp or pkg)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_ensure("pyyaml", "yaml")
_ensure("librosa")
_ensure("tqdm")

import numpy as np
import pandas as pd
import yaml
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import metrics
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch  :", torch.__version__)
print("librosa:", librosa.__version__)
print("device :", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU (fine for this small stage)")

## 2 · Path constants and run switches ← edit here to adjust

- **Read**: `STAGE1_DIR` (frozen baseline, for reference), `STAGE2_DIR` (Stage 2 full results, used in the final comparison)
- **Write**: `STAGE3_DIR`
- **Does not read** `BASELINE_DIR` — this stage involves no MIMII / pretrained weights

In [ ]:
# ============================ Edit area ============================
DRIVE_ROOT    = "/content/drive/MyDrive/HVAC_fan"

TARGET_ZIP    = f"{DRIVE_ROOT}/raw_audio.zip"

RESULTS_ROOT  = f"{DRIVE_ROOT}/HVAC2_Results"
STAGE1_DIR    = f"{RESULTS_ROOT}/Stage1"              # frozen baseline (read, for reference)
STAGE2_DIR    = f"{RESULTS_ROOT}/Stage2"              # Stage 2 full results (read, final comparison)
STAGE3_DIR    = f"{RESULTS_ROOT}/Stage3"              # this stage (write)

EXTRACT_ROOT  = "/content/data"                        # local disk: extraction and feature cache, shared with earlier stages

LIMIT_RUNS    = None        # None = run all 40; set a small number for a trial run
# ===============================================================

STAGE1_LOGS   = f"{STAGE1_DIR}/logs"
STAGE2_LOGS   = f"{STAGE2_DIR}/logs"
STAGE3_MODELS = f"{STAGE3_DIR}/models"
STAGE3_LOGS   = f"{STAGE3_DIR}/logs"
STAGE3_SCORES = f"{STAGE3_LOGS}/clip_scores"

RUNS_JSONL    = f"{STAGE3_LOGS}/stage3_runs.jsonl"
ERRORS_JSONL  = f"{STAGE3_LOGS}/stage3_errors.jsonl"

TARGET_EXTRACT = f"{EXTRACT_ROOT}/target"
TARGET_ROOT    = f"{TARGET_EXTRACT}/raw_audio"
FEATURE_DIR    = f"{EXTRACT_ROOT}/features"            # same cache dir as earlier stages

for label, p in [("target zip", TARGET_ZIP), ("Stage 1 out", STAGE1_DIR), ("Stage 2 out", STAGE2_DIR),
                 ("Stage 3 out", STAGE3_DIR), ("feat cache", FEATURE_DIR)]:
    print(f"  {label:<12} {p}")
print(f"  LIMIT_RUNS = {LIMIT_RUNS}")

## 3 · Config (`code/configs/config.yaml` embedded verbatim) + self-check + experiment enumeration

The same config and self-check as previous stages, plus verbatim copies of `rng_for` / `nested_order` / `nested_levels` (nested sampling),
`parse_clip` / `loro_split` / `target_finetune_sets` (LORO), and `enumerate_stage3` / `experiment_id` (experiment enumeration) from the repo's `code/configs/config.py`.

**LORO consistency is guaranteed by the functions themselves**: `loro_split` depends only on filenames and the fold/condition definitions in config,
and reads no external split file, so it naturally produces the same folds as Stages 1 and 2.

In [ ]:
_CONFIG_YAML_RAW = r"""
# ============================================================================
# HVAC 2.0 — experiment configuration (single source of truth)
# Matches proposal/proposal.md v6. On any conflict, the proposal wins and this file is updated.
# This file holds only "constants and rules"; no training logic.
# ============================================================================

project: HVAC_2.0
proposal_version: v6
seed: 42                      # the one random seed for the whole project (proposal §4.1)

# ---------------------------------------------------------------------------
# Paths. env selects which block is used; keep colab when running on Colab, set local for local debugging.
# ---------------------------------------------------------------------------
paths:
  env: colab                  # colab | local

  colab:
    drive_root: /content/drive/MyDrive
    # Source and target domains are both zips on Drive. Always extract to the Colab local disk /content/data,
    # never back onto Drive: Drive small-file IO is extremely slow, and it would push thousands of wavs to the cloud.
    # The local disk is ephemeral and must be re-extracted after a runtime restart -- extraction should be idempotent (skip if the directory exists).
    extract_root: /content/data

    # Source domain: MIMII 0dB fan
    mimii_zip: /content/drive/MyDrive/HVAC_fan/0_dB_fan.zip
    mimii_extract_to: /content/data/mimii
    mimii_root: /content/data/mimii/fan          # contains id_00/id_02/id_04/id_06

    # Target domain: 180 self-recorded fan clips
    target_zip: /content/drive/MyDrive/HVAC_fan/raw_audio.zip
    target_extract_to: /content/data/target
    target_root: /content/data/target/raw_audio  # contains {condition}/{voltage}/{noise}/
    recording_log: null                          # inventory check only; metadata comes from filenames

    # Outputs: written back to Drive so they survive runtime restarts
    outputs_root: /content/drive/MyDrive/HVAC_fan/HVAC2_Results

  local:
    drive_root: null
    extract_root: null
    # Already extracted locally; no zips needed
    mimii_zip: null
    mimii_extract_to: null
    mimii_root: data/mimii/fan
    target_zip: null
    target_extract_to: null
    target_root: data/raw_audio
    recording_log: data/metadata/recording_log.csv
    outputs_root: outputs

outputs:
  # Stages 2+3 total ~1032 train/eval runs. Weights are discarded after use; only CSVs are saved (proposal §7, stage 2)
  persist_finetuned_weights: false
  pretrained_models_dir: models        # relative to outputs_root; these 5 weights must be kept
  results_dir: logs
  figures_dir: figures

# ---------------------------------------------------------------------------
# Source domain: MIMII fan 0dB (proposal §4.1)
# ---------------------------------------------------------------------------
source_domain:
  name: MIMII_fan_0dB
  machine_ids: [id_00, id_02, id_04, id_06]

  # Official counts, used for a startup self-check: scanned file counts must match exactly, else fail immediately
  official_counts:
    id_00: {normal: 1011, abnormal: 407}
    id_02: {normal: 1016, abnormal: 359}
    id_04: {normal: 1033, abnormal: 348}
    id_06: {normal: 1015, abnormal: 361}

  # Evaluation set: class ratio 1:2, aligned with the target domain's 60:120 (proposal §3)
  eval_set:
    n_normal: 160
    n_abnormal: 320

  # Training pool = all normal clips of that machine ID − the 160 used by the evaluation set
  train_pool_size:
    id_00: 851
    id_02: 856
    id_04: 873
    id_06: 855

  # Merged model: equal contribution from each machine ID
  merged_model:
    # Evaluation set must be a subset of each ID's fixed evaluation set; no independent re-draw (proposal §4.1)
    eval_per_id: {normal: 40, abnormal: 80}
    # Training set: 215 clips drawn from each ID's "training pool", so the total matches the single-ID models
    train_per_id: 215
    train_total: 860

# ---------------------------------------------------------------------------
# Target domain: 180 self-recorded fan clips (proposal §4.2)
# ---------------------------------------------------------------------------
target_domain:
  name: self_recorded_fan
  device: Razer Seiren V3 Mini

  conditions: [normal, blocked, imbalance]
  normal_conditions: [normal]                 # 60 clips, the only part that may enter training
  abnormal_conditions: [blocked, imbalance]   # 120 clips, evaluation only
  voltages: ["4V", "8V", "12V"]
  noise_levels: [quiet, noise]
  n_runs: 10

  n_clips_total: 180
  n_normal_total: 60
  n_abnormal_total: 120

  # Filenames are the single source of truth for metadata; recording_log.csv is only an inventory check
  # (the csv has 206 rows with duplicates and misaligned columns; after dedup, the 180 filenames form a complete 18×10 grid)
  filename_pattern: "{condition}_{voltage}_{noise}_run{run:02d}.wav"
  relpath_pattern: "{condition}/{voltage}/{noise}/{filename}"

# ---------------------------------------------------------------------------
# Feature extraction (proposal §5, identical to baseline_runner.ipynb)
# ---------------------------------------------------------------------------
features:
  sr: 16000
  duration_sec: 10
  n_mels: 64
  frames: 5           # concatenate 5 adjacent frames
  n_fft: 1024
  hop_length: 512
  power: 2.0
  input_dim: 320      # = n_mels * frames
  # Each 10 s clip yields ~309 vectors of dim 320 (313 frames − 5 + 1)

# ---------------------------------------------------------------------------
# Model architecture (proposal §5)
# ---------------------------------------------------------------------------
model:
  arch: MIMII_Baseline_AE
  encoder_dims: [320, 64, 64, 8]
  decoder_dims: [8, 64, 64, 320]
  bottleneck_dim: 8
  activation: relu
  output_activation: none      # official baseline has no activation on the output layer
  # Layer indices inside the torch module, used by the layer schemes to freeze/unfreeze by name
  layer_index:
    encoder_linears: ["encoder.0", "encoder.2", "encoder.4"]   # 320→64, 64→64, 64→8
    decoder_linears: ["decoder.0", "decoder.2", "decoder.4"]   # 8→64, 64→64, 64→320

# ---------------------------------------------------------------------------
# Pretraining (stage 0, proposal §5 / §7 stage 0)
# ---------------------------------------------------------------------------
pretrain:
  optimizer: adam
  lr: 0.001            # torch Adam default, same as baseline_runner.ipynb
  loss: mse
  batch_size: 512
  epochs: 50
  validation_split: 0.1
  shuffle: true
  # Right after training, evaluate once on the model's own 480-clip source eval set and store as the "source baseline" (subtrahend for forgetting)
  record_source_baseline: true

# ---------------------------------------------------------------------------
# Fine-tuning (stage 2) — all four layer schemes share one hyperparameter set, not tuned per scheme (proposal §7, stage 2)
# ---------------------------------------------------------------------------
finetune:
  optimizer: adam
  loss: mse
  lr: 0.0001           # 1/10 of pretraining lr; at the 40-clip full level the movement budget is still < 2.5% of pretraining
  epochs: 30           # derived from a step budget: ~750 updates at the 5-clip level, ~5820 at the 40-clip level
  batch_size: 64       # 512 would give only 4 batches per epoch at the 5-clip level, too few for Adam momentum to warm up
  source_replay: false          # no source-domain replay (proposal §7, stage 2)
  early_stopping: false         # no early stopping; rationale in proposal §7, stage 2
  restart_from_pretrained_each_fold: true   # every LORO fold must restart from the pretrained weights

# ---------------------------------------------------------------------------
# LORO validation protocol (proposal §6)
# ---------------------------------------------------------------------------
loro:
  applies_to: large_shift_only     # no LORO for small shift (proposal §6.4)
  n_folds: 10                      # fold k holds out the whole run == k+1
  fold_key: run                    # determined by runNN in the filename
  clips_per_fold: 18               # one clip from each of the 18 conditions
  normals_per_fold: 6
  abnormals_per_fold: 12
  max_train_normals: 54            # 60 − 6, which is why the data levels stop at 40
  report: [pooled, per_fold]       # compute both views (proposal §6.2)
  per_fold_stats: [mean, std]
  pooled_normalization: none       # no per-fold normalization; listed as a limitation

# ---------------------------------------------------------------------------
# Experiment grid (proposal §7, stage 2)
# ---------------------------------------------------------------------------
grid:
  data_levels: [5, 10, 20, 40]

  # Nested sampling: 5 ⊂ 10 ⊂ 20 ⊂ 40 (proposal §4.3)
  nested_sampling:
    enabled: true
    rule: "Candidates are put in a fixed order by a fixed rule; level n takes the first n"
    # Ordering = sort by filename for a deterministic initial order, then shuffle with a numpy RNG
    canonical_order: filename_sort_then_shuffle
    rng: numpy_default_rng
    # Small shift: order each target ID's training pool once (seed scope includes only the ID)
    small_shift_seed_scope: [seed, target_id]
    # Large shift: available clips differ per fold, so each fold is ordered separately (seed scope includes the fold)
    large_shift_seed_scope: [seed, fold]

  layer_schemes:
    full:
      desc: Full fine-tuning
      trainable: ["encoder.0", "encoder.2", "encoder.4", "decoder.0", "decoder.2", "decoder.4"]
    decoder_only:
      desc: Decoder only
      trainable: ["decoder.0", "decoder.2", "decoder.4"]
    encoder_only:
      desc: Encoder only
      trainable: ["encoder.0", "encoder.2", "encoder.4"]
    bottleneck_only:
      desc: Bottleneck-adjacent layers only (the two layers into and out of the bottleneck, 64→8 and 8→64)
      trainable: ["encoder.4", "decoder.0"]

# ---------------------------------------------------------------------------
# Experiment enumeration per stage (counts are self-checked at startup by config.py; mismatch raises)
# ---------------------------------------------------------------------------
stages:
  stage0_pretrain:
    n_models: 5                  # 4 single-ID + 1 merged
  stage1_frozen_eval:
    small_shift_pairs: 12        # 4 starts × 3 unseen IDs each
    large_shift_models: 5        # all 5 models evaluated on large shift (incl. merged)
    large_shift_folds: 10
    shift_ratio:
      aggregation: median
      denominator: source_eval_normals          # the 160 normals of that ID's eval set
      numerator_small_shift: target_eval_normals # the 160 normals of the target ID's eval set
      numerator_large_shift: all_60_normals      # frozen eval has no leakage, so use all 60 clips
  stage2_finetune:
    small_shift:
      starts: 4
      targets_per_start: 3
      folds: 1
      n_configs: 192
      n_runs: 192
    large_shift:
      starts: 5                  # 4 single-ID + 1 merged
      targets_per_start: 1
      folds: 10
      n_configs: 80
      n_runs: 800
    n_configs_total: 272
    n_runs_total: 992
  stage3_no_pretrain:
    init: random
    layer_scheme: full           # random init has no "learned layers", so no layer-scheme variable
    n_configs: 4
    folds: 10
    n_runs: 40
    source_metrics: null         # no source domain; reported as N/A

total_runs: 1032                 # 992 + 40

# ---------------------------------------------------------------------------
# Evaluation metrics (proposal §3)
# ---------------------------------------------------------------------------
metrics:
  primary: auc                   # floor is always 0.5, independent of class ratio
  secondary: best_f1             # max over a scan of all thresholds
  best_f1_trivial_floor: 0.8     # at a 1:2 class ratio, "predict all abnormal" already gives 0.8
  n_threshold_scan_points: 1000  # matches the linspace in baseline_runner.ipynb
  clip_score: mean_mse_over_frames   # per clip = mean reconstruction MSE over all its feature vectors
  reconstruction_mse_aggregation: median   # threshold shift ratio uses the median
"""
CONFIG_YAML = _CONFIG_YAML_RAW.lstrip("\n")
CONFIG_SHA256 = hashlib.sha256(CONFIG_YAML.encode("utf-8")).hexdigest()
cfg = yaml.safe_load(CONFIG_YAML)


def self_check(cfg):
    src, tgt, grid, stages = cfg["source_domain"], cfg["target_domain"], cfg["grid"], cfg["stages"]
    n_eval_normal, n_eval_abnormal = src["eval_set"]["n_normal"], src["eval_set"]["n_abnormal"]
    src_ratio = n_eval_abnormal / n_eval_normal
    tgt_ratio = tgt["n_abnormal_total"] / tgt["n_normal_total"]
    assert abs(src_ratio - tgt_ratio) < 1e-9
    floor = 2 * src_ratio / (src_ratio + 1) / (1 + src_ratio / (src_ratio + 1))
    assert abs(floor - cfg["metrics"]["best_f1_trivial_floor"]) < 1e-9
    n_cells = len(tgt["conditions"]) * len(tgt["voltages"]) * len(tgt["noise_levels"])
    assert n_cells * tgt["n_runs"] == tgt["n_clips_total"] == 180
    n_normal_cells = len(tgt["normal_conditions"]) * len(tgt["voltages"]) * len(tgt["noise_levels"])
    assert n_normal_cells * tgt["n_runs"] == tgt["n_normal_total"]
    feat = cfg["features"]
    assert feat["n_mels"] * feat["frames"] == feat["input_dim"]
    assert cfg["model"]["encoder_dims"][0] == feat["input_dim"]
    loro = cfg["loro"]
    assert loro["n_folds"] == tgt["n_runs"]
    assert loro["clips_per_fold"] == n_cells
    assert loro["normals_per_fold"] == n_normal_cells
    assert loro["max_train_normals"] == tgt["n_normal_total"] - loro["normals_per_fold"]
    levels = grid["data_levels"]
    assert levels == sorted(set(levels)) and max(levels) <= loro["max_train_normals"]
    s3 = stages["stage3_no_pretrain"]
    assert s3["n_configs"] == len(levels)
    assert s3["n_runs"] == s3["n_configs"] * s3["folds"] == 40
    assert s3["folds"] == loro["n_folds"]
    assert s3["layer_scheme"] == "full"
    ft = cfg["finetune"]
    assert ft["lr"] and ft["epochs"] and ft["batch_size"]
    assert ft["early_stopping"] is False


self_check(cfg)

SEED    = cfg["seed"]
FEAT    = cfg["features"]
FT      = cfg["finetune"]
LORO    = cfg["loro"]
TGT     = cfg["target_domain"]
LEVELS  = cfg["grid"]["data_levels"]
N_SCAN  = cfg["metrics"]["n_threshold_scan_points"]
F1_FLOOR = cfg["metrics"]["best_f1_trivial_floor"]
S3      = cfg["stages"]["stage3_no_pretrain"]


# --- The functions below are verbatim copies from the repo's code/configs/config.py ---

def rng_for(seed, *parts):
    import zlib
    tag = "|".join(str(p) for p in parts).encode("utf-8")
    return np.random.default_rng([seed, zlib.crc32(tag)])


def nested_order(items, seed, *scope):
    ordered = sorted(items)
    idx = rng_for(seed, *scope).permutation(len(ordered))
    return [ordered[i] for i in idx]


def take_level(ordered, n):
    if n > len(ordered):
        raise ValueError(f"Level {n} exceeds available samples {len(ordered)}")
    return list(ordered[:n])


def nested_levels(items, levels, seed, *scope):
    ordered = nested_order(items, seed, *scope)
    out = {n: take_level(ordered, n) for n in levels}
    for smaller, larger in zip(levels, levels[1:]):
        assert out[smaller] == out[larger][:smaller], "Nesting broken"
    return out


CLIP_RE = re.compile(r"^(?P<condition>\w+?)_(?P<voltage>\d+V)_(?P<noise>\w+?)_run(?P<run>\d{2})\.wav$")


def parse_clip(filename):
    m = CLIP_RE.match(os.path.basename(filename))
    if not m:
        raise ValueError(f"Filename does not match convention: {filename}")
    d = m.groupdict()
    d["run"] = int(d["run"])
    return d


def loro_split(cfg, filenames, fold):
    n_folds = cfg["loro"]["n_folds"]
    if not 0 <= fold < n_folds:
        raise ValueError(f"fold must be in [0, {n_folds}), got {fold}")
    held_out_run = fold + 1
    test_clips, train_normals = [], []
    for f in filenames:
        meta = parse_clip(f)
        if meta["run"] == held_out_run:
            test_clips.append(f)
        elif meta["condition"] in cfg["target_domain"]["normal_conditions"]:
            train_normals.append(f)
    loro = cfg["loro"]
    assert len(test_clips) == loro["clips_per_fold"]
    assert len(train_normals) == loro["max_train_normals"]
    return sorted(train_normals), sorted(test_clips)


def target_finetune_sets(cfg, filenames, fold):
    train_normals, _ = loro_split(cfg, filenames, fold)
    scope = cfg["grid"]["nested_sampling"]["large_shift_seed_scope"]
    parts = [fold if s == "fold" else s for s in scope if s != "seed"]
    return nested_levels(train_normals, cfg["grid"]["data_levels"], cfg["seed"], "large_shift", *parts)


def enumerate_stage3(cfg):
    s3 = cfg["stages"]["stage3_no_pretrain"]
    return [dict(stage="stage3", shift="large", start="random_init", target="self_recorded_fan",
                data_level=n, layer_scheme=s3["layer_scheme"], n_folds=s3["folds"], source_eval=None)
            for n in cfg["grid"]["data_levels"]]


def experiment_id(c):
    return f"{c['stage']}_{c['shift']}_{c['start']}_to_{c['target']}_n{c['data_level']}_{c['layer_scheme']}"


CONFIGS = enumerate_stage3(cfg)
assert len(CONFIGS) == S3["n_configs"] == 4
assert sum(c["n_folds"] for c in CONFIGS) == S3["n_runs"] == 40

print(f"Config loaded (proposal {cfg['proposal_version']}), sha256 = {CONFIG_SHA256[:16]}...")
print(f"Self-check passed ✅")
print(f"  Data levels : {LEVELS} (only layer scheme: full; random init has no \"learned layers\")")
print(f"  Hyperparams : lr={FT['lr']} epochs={FT['epochs']} batch={FT['batch_size']} early_stop={FT['early_stopping']}")
print(f"  Configs     : {len(CONFIGS)}  Runs: {S3['n_runs']}")
for c in CONFIGS:
    print(f"    {experiment_id(c)}")

## 4 · Mount Drive, extract target domain, layout and grid checks

Only `raw_audio.zip` (target domain) is extracted. If the layout or counts don't match, the actual structure is printed and an error is raised.
Also checks that all 10 LORO folds are self-consistent (18 = 6 + 12, 54 clips on the training side).

In [ ]:
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

for d in (STAGE3_MODELS, STAGE3_LOGS, STAGE3_SCORES):
    os.makedirs(d, exist_ok=True)


def print_tree(base, max_depth=3, max_entries=12):
    base = os.path.abspath(base)
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        n_wav = sum(1 for f in files if f.lower().endswith(".wav"))
        print("  " * depth + (os.path.basename(root) or root) + "/" + (f"   [{n_wav} wav]" if n_wav else ""))
        dirs.sort()
        if len(dirs) > max_entries:
            dirs[:] = dirs[:max_entries]


def detect_dir(base, required_subdirs):
    for root, dirs, _ in os.walk(base):
        if all(os.path.isdir(os.path.join(root, s)) for s in required_subdirs):
            return root
    return None


if not os.path.exists(TARGET_ZIP):
    parent = os.path.dirname(TARGET_ZIP)
    raise FileNotFoundError(f"❌ Target-domain zip not found: {TARGET_ZIP}\n   Actual contents of {parent}:\n     "
                            + "\n     ".join(sorted(os.listdir(parent)) if os.path.isdir(parent) else ["(directory does not exist)"]))

if all(os.path.isdir(os.path.join(TARGET_ROOT, c)) for c in TGT["conditions"]):
    print(f"✅ Target domain already on local disk, skipping extraction")
else:
    os.makedirs(TARGET_EXTRACT, exist_ok=True)
    print(f"Extracting target domain ({os.path.getsize(TARGET_ZIP) / 1024**3:.2f} GB) → {TARGET_EXTRACT} ...")
    t0 = time.time()
    with zipfile.ZipFile(TARGET_ZIP) as zf:
        for m in tqdm(zf.infolist(), desc="unzip target", unit="file"):
            zf.extract(m, TARGET_EXTRACT)
    print(f"  Done in {time.time() - t0:.0f} s")

detected = detect_dir(TARGET_EXTRACT, TGT["conditions"])
if detected is None or os.path.realpath(detected) != os.path.realpath(TARGET_ROOT):
    print(f"❌ Target-domain directory layout mismatch. Expected {TARGET_ROOT}, detected {detected}")
    print_tree(TARGET_EXTRACT)
    raise RuntimeError("Path problem — set TARGET_ROOT to the detected value (section 2 cell) and re-run this cell.")

target_files = sorted(glob.glob(os.path.join(TARGET_ROOT, "**", "*.wav"), recursive=True), key=os.path.basename)
if len(target_files) != TGT["n_clips_total"]:
    print_tree(TARGET_EXTRACT)
    raise RuntimeError(f"Target domain should have {TGT['n_clips_total']} clips, found {len(target_files)} — data problem.")

grid_chk = defaultdict(list)
for f in target_files:
    d = parse_clip(f)
    grid_chk[(d["condition"], d["voltage"], d["noise"])].append(d["run"])
assert len(grid_chk) == LORO["clips_per_fold"] and all(sorted(v) == list(range(1, TGT["n_runs"] + 1)) for v in grid_chk.values()), \
    "18×10 grid incomplete — data problem, LORO cannot hold out whole runs"

TARGET_NAMES = [os.path.basename(f) for f in target_files]
for k in range(LORO["n_folds"]):
    loro_split(cfg, TARGET_NAMES, k)
print(f"✅ Target domain check passed: {TARGET_ROOT} ({len(target_files)} clips, 18×10 grid complete, 10 LORO folds consistent)")

## 5 · Feature cache (target domain only)

Shares the cache directory with previous stages; if the `target_all` cache from Stage 0/1/2 is still there (same runtime), this is instant.
Otherwise features are extracted fresh, ~20~60 seconds for 180 clips.

In [ ]:
os.makedirs(FEATURE_DIR, exist_ok=True)


def file_to_vector_array(file_name):
    n_mels, frames = FEAT["n_mels"], FEAT["frames"]
    n_fft, hop_length, power = FEAT["n_fft"], FEAT["hop_length"], FEAT["power"]
    y, sr = librosa.load(file_name, sr=None, mono=True)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, power=power)
    log_mel = 20.0 / power * np.log10(mel + sys.float_info.epsilon)
    size = log_mel.shape[1] - frames + 1
    if size < 1:
        return np.empty((0, n_mels * frames), float)
    out = np.zeros((size, n_mels * frames), float)
    for t in range(frames):
        out[:, n_mels * t: n_mels * (t + 1)] = log_mel[:, t: t + size].T
    return out


stem = os.path.join(FEATURE_DIR, "target_all")
npy_path, meta_path = stem + ".npy", stem + ".meta.json"
rel = [os.path.relpath(f, TARGET_ROOT) for f in target_files]
if os.path.exists(npy_path) and os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    reuse = set(rel) <= set(meta["files"])
else:
    reuse = False

t0 = time.time()
if not reuse:
    chunks, offsets = [], [0]
    for f in tqdm(target_files, desc="target", unit="clip"):
        v = file_to_vector_array(f).astype(np.float32)
        chunks.append(v)
        offsets.append(offsets[-1] + len(v))
    X = np.concatenate(chunks, axis=0)
    del chunks
    np.save(npy_path, X)
    meta = {"files": rel, "offsets": offsets, "n_clips": len(target_files), "n_vectors": int(X.shape[0]), "dim": int(X.shape[1])}
    with open(meta_path, "w") as f:
        json.dump(meta, f)
    del X

per_clip = {meta["offsets"][i + 1] - meta["offsets"][i] for i in range(len(meta["offsets"]) - 1)}
print(f"Set of vectors-per-clip values: {sorted(per_clip)}  (expected 309)")
print(f"✅ Features ready ({'cache reused' if reuse else 'freshly extracted'}) in {time.time() - t0:.0f} s")

_X_ALL = np.load(npy_path, mmap_mode="r")
_INDEX = {f: i for i, f in enumerate(meta["files"])}
_OFF = meta["offsets"]


def load_vectors(files):
    """files: paths relative to TARGET_ROOT. Returns (X float32, per-clip slices)."""
    parts, slices, cur = [], [], 0
    for f in files:
        i = _INDEX[f]
        seg = np.asarray(_X_ALL[_OFF[i]:_OFF[i + 1]])
        parts.append(seg)
        slices.append((cur, cur + len(seg)))
        cur += len(seg)
    return np.concatenate(parts, axis=0), slices

## 6 · Model, GPU-resident per-fold eval sets, training and evaluation functions

Model architecture is the same as Stage 0, **but every run is randomly initialized** (no `load_state_dict`).
The eval set is the 180 target-domain clips, split by fold into 10 subsets and loaded into GPU memory once.

`best_f1_scan` uses the same vectorized implementation as Stage 2 (cross-checked against the point-by-point loop; see the Stage 2 notebook).

In [ ]:
class MIMII_Baseline_AE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 8), nn.ReLU())
        self.decoder = nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, input_dim))

    def forward(self, x):
        return self.decoder(self.encoder(x))


class EvalSet:
    def __init__(self, X, slices, labels, files):
        self.X = torch.from_numpy(np.ascontiguousarray(X)).to(DEVICE)
        self.labels = np.asarray(labels)
        self.files = list(files)
        self.n = len(slices)
        idx = np.zeros(len(X), dtype=np.int64)
        for k, (a, b) in enumerate(slices):
            idx[a:b] = k
        self.clip_idx = torch.from_numpy(idx).to(DEVICE)
        self.counts = torch.bincount(self.clip_idx, minlength=self.n).double()

    def subset(self, mask):
        keep = np.where(mask)[0]
        remap = {old: new for new, old in enumerate(keep)}
        rows = torch.isin(self.clip_idx, torch.as_tensor(keep, device=DEVICE))
        sub = object.__new__(EvalSet)
        sub.X = self.X[rows]
        sub.labels = self.labels[keep]
        sub.files = [self.files[i] for i in keep]
        sub.n = len(keep)
        old_idx = self.clip_idx[rows].cpu().numpy()
        sub.clip_idx = torch.from_numpy(np.vectorize(remap.get)(old_idx)).to(DEVICE)
        sub.counts = torch.bincount(sub.clip_idx, minlength=sub.n).double()
        return sub


t0 = time.time()
Xt, st = load_vectors(rel)
TARGET_META = [parse_clip(f) for f in target_files]
TARGET_LABELS = [0 if d["condition"] in TGT["normal_conditions"] else 1 for d in TARGET_META]
FULL_EVAL = EvalSet(Xt, st, TARGET_LABELS, [os.path.basename(f) for f in target_files])
TARGET_RUNS = np.array([d["run"] for d in TARGET_META])
FOLD_EVAL = {k: FULL_EVAL.subset(TARGET_RUNS == k + 1) for k in range(LORO["n_folds"])}
for k, e in FOLD_EVAL.items():
    assert e.n == LORO["clips_per_fold"] and (e.labels == 0).sum() == LORO["normals_per_fold"]
del Xt
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
print(f"✅ Per-fold eval sets resident in GPU/RAM (10 folds × 18 clips) in {time.time() - t0:.0f} s")


def train_one(c, fold, seed):
    """Random init → train → return (model, info)."""
    torch.manual_seed(seed)
    model = MIMII_Baseline_AE(FEAT["input_dim"]).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=FT["lr"])
    criterion = nn.MSELoss()

    names = target_finetune_sets(cfg, TARGET_NAMES, fold)[c["data_level"]]
    base_to_rel = {os.path.basename(r): r for r in rel}          # filename → relative path in cache
    train_rel = [base_to_rel[n] for n in names]
    X_np, _ = load_vectors(train_rel)
    X = torch.from_numpy(np.ascontiguousarray(X_np)).to(DEVICE)

    n, bs, epochs = len(X), FT["batch_size"], FT["epochs"]
    n_batches = (n + bs - 1) // bs
    losses = []
    model.train()
    t0 = time.time()
    for epoch in range(epochs):
        idx = torch.randperm(n, device=DEVICE)
        total = 0.0
        for b in range(n_batches):
            xb = X[idx[b * bs:(b + 1) * bs]]
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), xb)
            loss.backward()
            optimizer.step()
            total += loss.item()
        losses.append(total / n_batches)
    train_sec = time.time() - t0
    model.eval()
    del X
    return model, {"n_train_clips": len(names), "n_train_vectors": n, "steps": n_batches * epochs,
                   "loss_first": losses[0], "loss_last": losses[-1], "loss_curve": [round(v, 6) for v in losses],
                   "train_sec": round(train_sec, 2)}


@torch.no_grad()
def score_evalset(model, ev, chunk=65536):
    mse = torch.empty(len(ev.X), device=DEVICE, dtype=torch.float64)
    for a in range(0, len(ev.X), chunk):
        xb = ev.X[a:a + chunk]
        mse[a:a + chunk] = torch.mean((xb - model(xb)) ** 2, dim=1).double()
    s = torch.zeros(ev.n, device=DEVICE, dtype=torch.float64).index_add_(0, ev.clip_idx, mse) / ev.counts
    return s.cpu().numpy()


def best_f1_scan(y_true, y_scores, n_points=None):
    n_points = N_SCAN if n_points is None else n_points
    y = np.asarray(y_true).astype(bool)
    s = np.asarray(y_scores, dtype=np.float64)
    ts = np.linspace(s.min(), s.max(), n_points)
    pred = s[None, :] > ts[:, None]
    tp = (pred & y).sum(1); fp = (pred & ~y).sum(1); fn = (~pred & y).sum(1); tn = (~pred & ~y).sum(1)
    denom = 2 * tp + fp + fn
    f1 = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    i = int(np.argmax(f1))
    if f1[i] <= 0:
        return {"f1": 0.0, "threshold": float("nan"), "precision": 0.0, "recall": 0.0, "accuracy": float(tn[0] + tp[0]) / len(s)}
    return {"f1": float(f1[i]), "threshold": float(ts[i]),
            "precision": float(tp[i] / (tp[i] + fp[i])) if tp[i] + fp[i] else 0.0,
            "recall": float(tp[i] / (tp[i] + fn[i])) if tp[i] + fn[i] else 0.0,
            "accuracy": float((tp[i] + tn[i]) / len(s))}


def evaluate_on(model, ev):
    scores = score_evalset(model, ev)
    return {"auc": float(metrics.roc_auc_score(ev.labels, scores)), **best_f1_scan(ev.labels, scores)}, scores


print(f"✅ Train/eval functions ready | params: {sum(p.numel() for p in MIMII_Baseline_AE(320).parameters()):,} (all trainable, none frozen)")

## 7 · Load Stage 1 frozen baseline (for reference)

Not required for this stage's computation; printed only to aid interpretation — the large-shift frozen AUC (pretrained, not fine-tuned)
can be read alongside this stage (no pretraining, 30 epochs of fine-tuning); together the three form a complete control chain:
**frozen (Stage 1) → fine-tuning without pretraining (Stage 3) → pretraining + fine-tuning (Stage 2)**.

In [ ]:
s1_path = os.path.join(STAGE1_LOGS, "stage1_large_shift.csv")
if os.path.exists(s1_path):
    s1 = pd.read_csv(s1_path)
    print(f"Stage 1 frozen baseline (large shift, mean pooled AUC of 5 pretrained models): {s1['pooled_auc'].mean():.4f}")
else:
    print(f"⚠️ {s1_path} not found, skipping (does not affect training; only one reference line is missing)")

## 8 · Run plan and checkpoint detection

The 40 runs are expanded into `run_id = {experiment_id}__fold{k}`. Same mechanism as Stage 2:
read `stage3_runs.jsonl` to find completed runs, queue the rest in order; a single failure is logged and the loop continues; re-running fills in the gaps automatically.

In [ ]:
def run_id_of(c, fold):
    return f"{experiment_id(c)}__fold{fold}"


PLAN = [(c, k) for c in CONFIGS for k in range(c["n_folds"])]
assert len(PLAN) == S3["n_runs"] == 40


def load_done():
    done = {}
    if os.path.exists(RUNS_JSONL):
        with open(RUNS_JSONL) as f:
            for ln, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    r = json.loads(line)
                    done[r["run_id"]] = r
                except json.JSONDecodeError:
                    print(f"  ⚠️ jsonl line {ln} corrupted, skipped; that run will be re-run")
    return done


DONE = load_done()
TODO = [(c, k) for c, k in PLAN if run_id_of(c, k) not in DONE]
n_err_prev = sum(1 for _ in open(ERRORS_JSONL)) if os.path.exists(ERRORS_JSONL) else 0

print(f"Planned {len(PLAN)} | done {len(DONE)} | to run {len(TODO)}"
      + (f" | session limit LIMIT_RUNS={LIMIT_RUNS}" if LIMIT_RUNS else "")
      + (f" | {n_err_prev} past failure records (will be retried automatically)" if n_err_prev else ""))
if TODO:
    print(f"Next: {run_id_of(*TODO[0])}")
else:
    print("🎉 All done, just run the section 10 comparison table")

## 9 · Main loop (40 runs)

Small scale, normally finishes in a few minutes, but the protection logic is the same as Stage 2: write scores first, then the jsonl line;
a single failure is logged to `stage3_errors.jsonl` and the loop continues; re-running this notebook resumes and retries automatically.

One line is printed per config (after all 10 folds): per-fold AUC mean±std, per-fold F1 mean±std.

In [ ]:
def append_jsonl(path, obj):
    with open(path, "a") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def write_clip_scores(eid, run_id, fold, rows):
    p = os.path.join(STAGE3_SCORES, f"{eid}.csv")
    header = "run_id,fold,file,label,score\n"
    if os.path.exists(p):
        with open(p) as f:
            old = [ln for ln in f.read().splitlines()[1:] if ln and not ln.startswith(run_id + ",")]
    else:
        old = []
    with open(p, "w") as f:
        f.write(header)
        for ln in old:
            f.write(ln + "\n")
        for fn, y, sc in rows:
            f.write(f"{run_id},{fold},{fn},{int(y)},{sc:.10e}\n")
        f.flush()
        os.fsync(f.fileno())


def do_run(c, fold):
    eid, rid = experiment_id(c), run_id_of(c, fold)
    seed = int(rng_for(SEED, "stage3", eid, fold).integers(0, 2**31 - 1))
    t0 = time.time()

    model, info = train_one(c, fold, seed)
    tgt, tgt_scores = evaluate_on(model, FOLD_EVAL[fold])

    rows = list(zip(FOLD_EVAL[fold].files, FOLD_EVAL[fold].labels, tgt_scores))
    write_clip_scores(eid, rid, fold, rows)

    torch.save({"model_state_dict": model.state_dict(), "input_dim": FEAT["input_dim"], "config": c,
                "fold": fold, "seed": seed, "config_sha256": CONFIG_SHA256},
               os.path.join(STAGE3_MODELS, f"{eid}__fold{fold}.pth"))

    rec = {
        "run_id": rid, "experiment_id": eid, "fold": fold, "data_level": c["data_level"], "seed": seed,
        **{k: v for k, v in info.items() if k != "loss_curve"}, "loss_curve": info["loss_curve"],
        "target_auc": tgt["auc"], "target_best_f1": tgt["f1"], "target_threshold": tgt["threshold"],
        "target_precision": tgt["precision"], "target_recall": tgt["recall"],
        "source_auc": None, "source_best_f1": None,          # no source domain: never saw MIMII
        "total_sec": round(time.time() - t0, 2), "finished_at": datetime.now().isoformat(timespec="seconds"),
    }
    append_jsonl(RUNS_JSONL, rec)
    del model
    return rec


def fmt_eta(sec):
    return str(timedelta(seconds=int(sec)))


queue = TODO[:LIMIT_RUNS] if LIMIT_RUNS else TODO
n_ok = n_fail = 0
recent = []
session_t0 = time.time()
config_runs = defaultdict(list)
for rid, r in DONE.items():
    config_runs[r["experiment_id"]].append(r)
configs_done_before = sum(1 for c in CONFIGS if len(config_runs[experiment_id(c)]) == c["n_folds"])
configs_done = 0

print(f"Start: {len(queue)} to run, {len(DONE)} already done | {datetime.now().strftime('%H:%M:%S')}")
bar = tqdm(total=len(PLAN), initial=len(DONE), unit="run", dynamic_ncols=True)

for c, fold in queue:
    eid, rid = experiment_id(c), run_id_of(c, fold)
    t0 = time.time()
    try:
        rec = do_run(c, fold)
        n_ok += 1
        DONE[rid] = rec
        config_runs[eid].append(rec)
    except Exception as e:
        n_fail += 1
        append_jsonl(ERRORS_JSONL, {"run_id": rid, "experiment_id": eid, "fold": fold, "error": repr(e),
                                    "traceback": traceback.format_exc()[-2000:],
                                    "at": datetime.now().isoformat(timespec="seconds")})
        print(f"  ❌ {rid} failed: {repr(e)[:120]} (logged, continuing)")
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        bar.update(1)
        continue

    dt = time.time() - t0
    recent = (recent + [dt])[-50:]
    remaining = len(queue) - n_ok - n_fail
    eta = np.mean(recent) * remaining if recent else 0
    bar.update(1)
    bar.set_postfix_str(f"{dt:.1f}s/run  ETA {fmt_eta(eta)}  failed {n_fail}")

    if len(config_runs[eid]) == c["n_folds"]:
        configs_done += 1
        rs = config_runs[eid]
        aucs = [r["target_auc"] for r in rs]
        f1s = [r["target_best_f1"] for r in rs]
        print(f"  ✅ [{configs_done_before + configs_done}/4] n={c['data_level']:<2} full "
              f"| fold AUC {np.mean(aucs):.4f}±{np.std(aucs):.4f} | fold F1 {np.mean(f1s):.4f}±{np.std(f1s):.4f} "
              f"| {sum(r['total_sec'] for r in rs):.1f}s")

bar.close()
print(f"\nThis session: ok {n_ok} | failed {n_fail} | elapsed {fmt_eta(time.time() - session_t0)}")
print(f"Total completed {len(DONE)}/{len(PLAN)}" + ("  🎉 All done" if len(DONE) == len(PLAN) else
      f"  ← {len(PLAN) - len(DONE)} remaining; re-running this notebook resumes automatically"))
if n_fail:
    print(f"⚠️ {n_fail} failures, see {ERRORS_JSONL}; re-running this notebook retries them automatically")

## 10 · Comparison table: no pretraining vs pretraining (same data level, same full scheme)

Core output required by `proposal/proposal.md`: validate the value of pretraining itself.

- **No-pretraining AUC**: this stage's pooled metric (scores of the 180 clips across 10 folds concatenated into one AUC)
- **Pretrained AUC**: mean pooled AUC, at the same data level, over the 5 starts (4 single-ID + merged) with `shift=large, layer_scheme=full`
  in Stage 2 `Stage2/logs/stage2_configs.csv`

Reading note: the best-F1 floor is 0.800; this setting is especially prone to being misread via F1 — always judge by AUC.

In [ ]:
DONE = load_done()
runs_df = pd.DataFrame([{k: v for k, v in r.items() if k != "loss_curve"} for r in DONE.values()])
if not runs_df.empty:
    runs_df = runs_df.sort_values("run_id")
    runs_df.to_csv(os.path.join(STAGE3_LOGS, "stage3_runs.csv"), index=False)

rows, incomplete = [], []
for c in CONFIGS:
    eid = experiment_id(c)
    rs = [r for r in DONE.values() if r["experiment_id"] == eid]
    if len(rs) != c["n_folds"]:
        incomplete.append((eid, len(rs), c["n_folds"]))
        continue
    sc = pd.read_csv(os.path.join(STAGE3_SCORES, f"{eid}.csv")).drop_duplicates(subset=["run_id", "file"], keep="last")
    assert len(sc) == TGT["n_clips_total"] and sc["fold"].nunique() == LORO["n_folds"], f"{eid} pooled scores incomplete: {len(sc)} rows"
    pooled_auc = float(metrics.roc_auc_score(sc["label"], sc["score"]))
    pooled_f1 = best_f1_scan(sc["label"].values, sc["score"].values)["f1"]
    fa = [r["target_auc"] for r in rs]; ff = [r["target_best_f1"] for r in rs]
    rows.append({"data_level": c["data_level"], "pooled_auc": pooled_auc, "pooled_best_f1": pooled_f1,
                "fold_auc_mean": np.mean(fa), "fold_auc_std": np.std(fa),
                "fold_f1_mean": np.mean(ff), "fold_f1_std": np.std(ff),
                "loss_first_mean": np.mean([r["loss_first"] for r in rs]),
                "loss_last_mean": np.mean([r["loss_last"] for r in rs])})

no_pt = pd.DataFrame(rows).sort_values("data_level") if rows else pd.DataFrame()
if incomplete:
    print(f"⚠️ {len(incomplete)} configs still incomplete; the comparison table will contain completed levels only: {incomplete}")
no_pt.to_csv(os.path.join(STAGE3_LOGS, "stage3_configs.csv"), index=False)

s2_path = os.path.join(STAGE2_LOGS, "stage2_configs.csv")
if not os.path.exists(s2_path) or no_pt.empty:
    print(f"⚠️ Missing {'Stage 2 results' if not os.path.exists(s2_path) else 'Stage 3 results'}, cannot build the full comparison table")
    no_pt
else:
    s2 = pd.read_csv(s2_path)
    s2_full = s2[(s2["shift"] == "large") & (s2["layer_scheme"] == "full")]
    with_pt = s2_full.groupby("data_level").agg(
        pretrained_auc=("pooled_auc", "mean"), pretrained_auc_std=("pooled_auc", "std"),
        pretrained_f1=("pooled_best_f1", "mean"), n_starts=("start", "count")).reset_index()

    cmp = no_pt.merge(with_pt, on="data_level", how="outer").sort_values("data_level")
    cmp["auc_diff"] = cmp["pretrained_auc"] - cmp["pooled_auc"]
    cmp["f1_diff"] = cmp["pretrained_f1"] - cmp["pooled_best_f1"]
    cmp.to_csv(os.path.join(STAGE3_LOGS, "stage3_vs_stage2_comparison.csv"), index=False)

    print("=" * 100)
    print("Table · No pretraining (Stage 3) vs pretraining + full fine-tuning (Stage 2, mean of 5 starts)")
    print("=" * 100)
    print(f"{'Level':<8}{'No-PT AUC':>14}{'PT AUC':>14}{'Diff':>10}{'  ':>2}"
          f"{'No-PT F1':>12}{'PT F1':>12}{'F1 diff':>10}")
    print("-" * 100)
    for r in cmp.itertuples():
        print(f"{r.data_level:<8}{r.pooled_auc:>14.4f}{r.pretrained_auc:>14.4f}{r.auc_diff:>+10.4f}{'  ':>2}"
              f"{r.pooled_best_f1:>12.4f}{r.pretrained_f1:>12.4f}{r.f1_diff:>+10.4f}")
    print("=" * 100)
    print(f"\n(F1 trivial floor = {F1_FLOOR}; both sides may sit at this value — judge the value of pretraining by the AUC columns, not F1)")
    print(f"\n✅ Stage 3 done. Outputs in {STAGE3_LOGS}/: stage3_runs.jsonl (source of truth), stage3_runs.csv, "
          f"stage3_configs.csv, stage3_vs_stage2_comparison.csv, clip_scores/ ({len(os.listdir(STAGE3_SCORES))} files)")
    print(f"Weights ({STAGE3_MODELS}): {len(glob.glob(os.path.join(STAGE3_MODELS, '*.pth')))} files")
    cmp